<a href="https://colab.research.google.com/github/ancestor9/Scikit-learn-model-selection-evaluation/blob/main/optuna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install optuna

In [2]:
# @title **1. Obtain Data & Split Data into Train & Test**
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

cancer = load_breast_cancer()
cancer = pd.DataFrame(np.c_[cancer['target'], cancer['data']],
                      columns = np.append('target', cancer['feature_names']))
cancer['target'] = cancer['target'].map({0.0: 1, 1.0:0}).astype('int64')

# Separate the independent variables from the dependent variable.
X = cancer.drop('target', axis = 1)
y = cancer['target']

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    stratify=y,
                                                    random_state=0)

**Optuna의 기본 알고리즘인 TPE (Tree-structured Parzen Estimator)라는 베이지안 최적화(Bayesian Optimization) 기반의 탐색 방식**

In [3]:
# @title **2. Optuna Optimization**

import optuna
import sklearn.datasets
import sklearn.ensemble
import sklearn.model_selection
"""
Optuna example that optimizes a classifier configuration for Iris dataset using sklearn.

In this example, we optimize a classifier configuration for Iris dataset. Classifiers are from
scikit-learn. We optimize both the choice of classifier (among SVC and RandomForest) and their
hyperparameters.

"""
def objective(trial):
    # 1. 랜덤 포레스트의 주요 하이퍼파라미터들 정의
    n_estimators = trial.suggest_int("n_estimators", 50, 500, step=50)  # 생성할 트리의 개수
    max_depth = trial.suggest_int("max_depth", 3, 30)  # 트리의 최대 깊이 (None 대신 상한선 설정 가능)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)  # 노드를 분할하기 위한 최소 샘플 수
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)  # 리프 노드에 있어야 할 최소 샘플 수
    max_features = trial.suggest_categorical("max_features", ["sqrt", "log2", None])  # 각 노드에서 사용할 피처 수의 방식
    bootstrap = trial.suggest_categorical("bootstrap", [True, False])  # 부트스트랩 샘플 사용 여부
    criterion = trial.suggest_categorical("criterion", ["gini", "entropy", "log_loss"])  # 분할 품질 측정 기준

    # 2. 파라미터를 적용한 랜덤 포레스트 모델 생성
    classifier_obj = sklearn.ensemble.RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        bootstrap=bootstrap,
        criterion=criterion,
        random_state=42,
        n_jobs=-1
    )

    # 3. 교차 검증 점수 계산
    score = sklearn.model_selection.cross_val_score(
        classifier_obj, X, y, n_jobs=-1, cv=3
    )
    accuracy = score.mean()
    return accuracy


if __name__ == "__main__":
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=100)

    # 4. 결과 확인
    print("Best Params:", study.best_params)

    # 5. **kwargs를 이용해 최적 파라미터로 최종 모델 생성 및 학습
    best_params = study.best_params
    best_rf_model = sklearn.ensemble.RandomForestClassifier(
        **best_params,
        random_state=42,
        n_jobs=-1
    )

    best_rf_model.fit(X, y)

[I 2026-11-12 04:44:01,237] A new study created in memory with name: no-name-c583044e-7a39-4854-b9aa-05fd4ceac251
[I 2026-11-12 04:44:06,791] Trial 0 finished with value: 0.9508029332590736 and parameters: {'n_estimators': 450, 'max_depth': 17, 'min_samples_split': 7, 'min_samples_leaf': 10, 'max_features': 'log2', 'bootstrap': True, 'criterion': 'gini'}. Best is trial 0 with value: 0.9508029332590736.
[I 2026-11-12 04:44:09,163] Trial 1 finished with value: 0.9543209876543209 and parameters: {'n_estimators': 300, 'max_depth': 15, 'min_samples_split': 7, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'bootstrap': True, 'criterion': 'gini'}. Best is trial 1 with value: 0.9543209876543209.
[I 2026-11-12 04:44:12,913] Trial 2 finished with value: 0.9630929174788824 and parameters: {'n_estimators': 300, 'max_depth': 11, 'min_samples_split': 6, 'min_samples_leaf': 5, 'max_features': 'log2', 'bootstrap': False, 'criterion': 'entropy'}. Best is trial 2 with value: 0.9630929174788824.
[I 2026-

Best Params: {'n_estimators': 50, 'max_depth': 13, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': 'log2', 'bootstrap': False, 'criterion': 'entropy'}


In [4]:
best_rf_model

RandomForestClassifier(bootstrap=False, criterion='entropy', max_depth=13,
                       max_features='log2', min_samples_leaf=3,
                       min_samples_split=5, n_estimators=50, n_jobs=-1,
                       random_state=42)

In [5]:
# @title **예측모형의 성능지표(검증데이터) 산출 사용자정의 함수**
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score,
    ConfusionMatrixDisplay
)


def evaluate_model(model, X_test, y_test, model_name="Model"):
    """
    모델과 테스트 데이터를 받아 주요 성능 지표를 계산하고
    컬럼 형태의 Pandas DataFrame으로 반환하는 함수
    """
    # 1. 예측값 및 확률값 계산
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    # 2. 혼동 행렬 및 오류 개수 계산
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    total_errors = fp + fn  # 오탐(틀린 개수) 총합

    # 3. 주요 지표 계산
    metrics = {
        "Accuracy (정확도)": accuracy_score(y_test, y_pred),
        "Precision (정밀도)": precision_score(y_test, y_pred),
        "Recall (재현율)": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "False Positives (FP)": int(fp),
        "False Negatives (FN)": int(fn),
        "Total Errors (FP+FN)": int(total_errors),
        "AUROC": roc_auc_score(y_test, y_proba),
        "PR AUC": average_precision_score(y_test, y_proba)
    }

    # 4. 데이터프레임으로 변환 후 소수점 포맷팅
    df = pd.DataFrame(list(metrics.items()), columns=["Metric", model_name])

    # 실수형 지표만 소수점 4자리로 포맷팅
    df[model_name] = df[model_name].apply(
        lambda x: f"{x:.4f}" if isinstance(x, float) else x
    )

    return df

In [6]:
# @title **모델 평가 및 기준 표 생성**
comparison_df = evaluate_model(best_rf_model, X_test, y_test, model_name="RF_Optuna")
comparison_df

,Metric,RF_Optuna
0,Accuracy (정확도),1.0000
1,Precision (정밀도),1.0000
2,Recall (재현율),1.0000
3,F1 Score,1.0000
4,False Positives (FP),0.0000
5,False Negatives (FN),0.0000
6,Total Errors (FP+FN),0.0000
7,AUROC,1.0000
8,PR AUC,1.0000


In [7]:
# @title **2. Scikit-Learn Cross Validation & Hyperparametertunning**
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
import scipy.stats as stats

# 1. 탐색할 하이퍼파라미터 분포 및 후보군 정의 (Optuna 분포와 동일한 범위)
param_distributions = {
    "n_estimators": [50, 100, 150, 200, 250, 300, 350, 400, 450, 500],
    "max_depth": stats.randint(3, 31),
    "min_samples_split": stats.randint(2, 11),
    "min_samples_leaf": stats.randint(1, 11),
    "max_features": ["sqrt", "log2", None],
    "bootstrap": [True, False],
    "criterion": ["gini", "entropy", "log_loss"]
}

# 2. 기본 모델 객체 생성
rf = RandomForestClassifier(random_state=42, n_jobs=-1)

# 3. RandomizedSearchCV 설정
# n_iter=100은 Optuna의 n_trials=100과 동일하게 100번의 조합을 시도합니다.
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=100,
    cv=3,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1
)

# 4. 탐색 실행
random_search.fit(X_train, y_train)

# 5. 결과 확인 및 최적 모델 추출
print("Best Params:", random_search.best_params_)
print("Best Score:", random_search.best_score_)

random_search.best_estimator_

Best Params: {'bootstrap': True, 'criterion': 'log_loss', 'max_depth': 11, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 8, 'n_estimators': 350}
Best Score: 0.9671361502347416


RandomForestClassifier(criterion='log_loss', max_depth=11, max_features='log2',
                       min_samples_split=8, n_estimators=350, n_jobs=-1,
                       random_state=42)

In [8]:
rf_metrics = evaluate_model(random_search.best_estimator_, X_test, y_test, model_name="RF_Scikit-Learn")
comparison_df["RF_Scikit-Learn"] = rf_metrics["RF_Scikit-Learn"]
comparison_df

,Metric,RF_Optuna,RF_Scikit-Learn
0,Accuracy (정확도),1.0000,0.9510
1,Precision (정밀도),1.0000,0.9107
2,Recall (재현율),1.0000,0.9623
3,F1 Score,1.0000,0.9358
4,False Positives (FP),0.0000,5.0000
5,False Negatives (FN),0.0000,2.0000
6,Total Errors (FP+FN),0.0000,7.0000
7,AUROC,1.0000,0.9818
8,PR AUC,1.0000,0.9861


### **주요 모델 default로 학습하고 예측하기**

In [9]:
!pip install catboost

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from tqdm.auto import tqdm

models = {
    "LogisticRegression": LogisticRegression(random_state=42, max_iter=1000),
    "SVC": SVC(probability=True, random_state=42),
    "KNN": KNeighborsClassifier(n_jobs=-1),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "GradientBoosting": GradientBoostingClassifier(random_state=42),
    "XGBoost": XGBClassifier(random_state=42, n_jobs=-1, eval_metric='logloss'),
    "LightGBM": LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1),
    "CatBoost": CatBoostClassifier(random_state=42, verbose=0)
}

for model_name, model in tqdm(list(models.items())):
    print(f'{model_name}학습하고 검증데이터 예측성능')
    model.fit(X_train, y_train)
    model_metrics = evaluate_model(model, X_test, y_test, model_name=model_name)
    comparison_df[model_name] = model_metrics[model_name]

comparison_df

  0%|          | 0/8 [00:00<?, ?it/s]

LogisticRegression학습하고 검증데이터 예측성능


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


SVC학습하고 검증데이터 예측성능
KNN학습하고 검증데이터 예측성능
DecisionTree학습하고 검증데이터 예측성능
GradientBoosting학습하고 검증데이터 예측성능
XGBoost학습하고 검증데이터 예측성능
LightGBM학습하고 검증데이터 예측성능
CatBoost학습하고 검증데이터 예측성능


,Metric,RF_Optuna,RF_Scikit-Learn,LogisticRegression,SVC,KNN,DecisionTree,GradientBoosting,XGBoost,LightGBM,CatBoost
0,Accuracy (정확도),1.0000,0.9510,0.9650,0.9301,0.9231,0.9510,0.9720,0.9580,0.9650,0.9650
1,Precision (정밀도),1.0000,0.9107,0.9444,0.9778,0.9038,0.8966,0.9455,0.9273,0.9444,0.9286
2,Recall (재현율),1.0000,0.9623,0.9623,0.8302,0.8868,0.9811,0.9811,0.9623,0.9623,0.9811
3,F1 Score,1.0000,0.9358,0.9533,0.8980,0.8952,0.9369,0.9630,0.9444,0.9533,0.9541
4,False Positives (FP),0.0000,5.0000,3.0000,1.0000,5.0000,6.0000,3.0000,4.0000,3.0000,4.0000
5,False Negatives (FN),0.0000,2.0000,2.0000,9.0000,6.0000,1.0000,1.0000,2.0000,2.0000,1.0000
6,Total Errors (FP+FN),0.0000,7.0000,5.0000,10.0000,11.0000,7.0000,4.0000,6.0000,5.0000,5.0000
7,AUROC,1.0000,0.9818,0.9945,0.9776,0.9593,0.9572,0.9918,0.9925,0.9916,0.9910
8,PR AUC,1.0000,0.9861,0.9913,0.9690,0.9496,0.8866,0.9912,0.9909,0.9905,0.9904
